# Laboratorio: determinante e invertibilidad

Este laboratorio compara tres perspectivas: expansión de Laplace, eliminación y geometría. Todos los cálculos algebraicos se mantienen exactos.

In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

sp.init_printing()

## 1. Determinante por eliminación

La función siguiente usa reemplazos de filas, que no alteran el determinante, e intercambios, que cambian su signo. No normaliza los pivotes; al final multiplica la diagonal de la matriz triangular.

In [ ]:
def determinante_por_eliminacion(A):
    U = sp.Matrix(A)
    if U.rows != U.cols:
        raise ValueError("El determinante requiere una matriz cuadrada")

    signo = 1
    pasos = []
    n = U.rows

    for columna in range(n):
        pivote = next((i for i in range(columna, n)
                       if U[i, columna] != 0), None)
        if pivote is None:
            return sp.Integer(0), pasos, U

        if pivote != columna:
            U.row_swap(pivote, columna)
            signo *= -1
            pasos.append((f"F{columna+1} <-> F{pivote+1}", U.copy()))

        for i in range(columna + 1, n):
            if U[i, columna] == 0:
                continue
            factor = sp.simplify(U[i, columna] / U[columna, columna])
            U.row_op(i, lambda valor, j: sp.simplify(
                valor - factor * U[columna, j]
            ))
            pasos.append((f"F{i+1} <- F{i+1} - ({factor})F{columna+1}",
                          U.copy()))

    determinante = sp.simplify(signo * sp.prod(U[i, i] for i in range(n)))
    return determinante, pasos, U

In [ ]:
A = sp.Matrix([
    [2, -1, 0],
    [3, 4, 1],
    [0, 5, 2],
])
det_A, pasos_A, U_A = determinante_por_eliminacion(A)

display(A)
for numero, (operacion, matriz) in enumerate(pasos_A, 1):
    print(f"Paso {numero}: {operacion}")
    display(matriz)
print("matriz triangular:")
display(U_A)
print("determinante:", det_A)

assert det_A == A.det()
assert det_A == 12

## 2. Verificación de las reglas por filas

Comprobaremos en una misma matriz el intercambio, el escalamiento y el reemplazo de filas.

In [ ]:
B = sp.Matrix([[1, 2, 0], [0, 3, 1], [2, 1, 4]])
d = B.det()

B_intercambio = B.copy()
B_intercambio.row_swap(0, 1)

B_escala = B.copy()
B_escala.row_op(1, lambda valor, j: 5 * valor)

B_reemplazo = B.copy()
B_reemplazo.row_op(2, lambda valor, j: valor - 7 * B_reemplazo[0, j])

print("det(B) =", d)
print("intercambio:", B_intercambio.det())
print("escalamiento:", B_escala.det())
print("reemplazo:", B_reemplazo.det())

assert B_intercambio.det() == -d
assert B_escala.det() == 5 * d
assert B_reemplazo.det() == d

## 3. Invertibilidad con parámetro

Para decidir cuándo una matriz simbólica es invertible, factorizamos su determinante. Los valores que anulan alguno de los factores son exactamente los valores singulares.

In [ ]:
a = sp.symbols("a", real=True)
A_a = sp.Matrix([
    [1, a, 1, 3],
    [a, 1, 3, 1],
    [1, 3, 1, a],
    [3, 1, a, 1],
])
det_factorizado = sp.factor(A_a.det())
display(det_factorizado)

raices = sp.solve(sp.Eq(det_factorizado, 0), a)
print("valores singulares:", raices)
assert det_factorizado == (a - 3)**2 * (a + 1) * (a + 5)
assert set(raices) == {-5, -1, 3}

## 4. Área y orientación en $\mathbb R^2$

Las columnas $u,v$ de una matriz $2\times2$ generan un paralelogramo de área $|\det[u\ v]|$. El signo indica si el par conserva o invierte la orientación canónica.

In [ ]:
u = np.array([3.0, 1.0])
v = np.array([1.0, 2.0])
C = np.column_stack([u, v])
area = abs(np.linalg.det(C))

vertices = np.array([[0, 0], u, u + v, v, [0, 0]])
fig, ax = plt.subplots(figsize=(6, 5))
ax.fill(vertices[:, 0], vertices[:, 1], alpha=0.25, color="tab:blue")
ax.plot(vertices[:, 0], vertices[:, 1], color="tab:blue")
ax.quiver([0, 0], [0, 0], [u[0], v[0]], [u[1], v[1]],
          angles="xy", scale_units="xy", scale=1,
          color=["crimson", "darkgreen"])
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)
ax.set(xlim=(-0.5, 5), ylim=(-0.5, 4),
       title=f"Área = |det(C)| = {area:.0f}")
plt.show()

assert sp.Matrix(C.astype(int)).det() == 5

## 5. Matriz idempotente

Una matriz idempotente satisface $P^2=P$. Si además fuera invertible, al multiplicar por $P^{-1}$ se obtendría $P=I$. Por tanto, toda matriz idempotente distinta de la identidad es singular.

In [ ]:
P = sp.Matrix([
    [2, -2, -4],
    [-1, 3, 4],
    [1, -2, -3],
])
display(P, P**2)
print("det(P) =", P.det())
print("rango(P) =", P.rank())

assert P**2 == P
assert P != sp.eye(3)
assert P.det() == 0

## 6. Ejercicios de laboratorio

1. Modifique la función de eliminación para devolver también el número de intercambios y la lista de pivotes.
2. Compare el tiempo de `det()` y de una expansión de Laplace programada recursivamente para matrices aleatorias de órdenes $2$ a $8$.
3. Construya una matriz cuyas columnas sean casi paralelas. Compare su determinante exacto con el cálculo decimal y discuta sensibilidad numérica.
4. Para cada valor singular $a\in\{-5,-1,3\}$ de $A(a)$, calcule rango y una base del núcleo.
5. Genere matrices ortogonales $2\times2$ de rotación y reflexión, y verifique que sus determinantes son $1$ y $-1$, respectivamente.